# cuDNN ResNet-18 Benchmark

This notebook is the presentation-friendly path through the cuDNN demo. It verifies cuDNN through PyTorch, runs a Conv2d microbenchmark, runs a ResNet-18 inference benchmark, and plots cross-platform results from Brev L4 and DGX Spark.

## 1. Context

- cuDNN provides optimized GPU primitives for deep learning.
- Frameworks such as PyTorch call cuDNN under the hood for operations like convolution, pooling, normalization, activation, and fused graph execution.
- For an SA, the value story is simple: existing framework code can get NVIDIA GPU acceleration with very little application rewrite.

In [ ]:
from pathlib import Path
import os

for candidate in [Path.cwd(), Path.cwd() / 'cuDNN' / 'examples', Path.cwd() / 'examples']:
    if (candidate / 'benchmark_cudnn.py').exists():
        os.chdir(candidate)
        break

print('Notebook working directory:', Path.cwd())

## 2. Setup & Sanity Check

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('cuDNN version:', torch.backends.cudnn.version())
print('cuDNN available:', torch.backends.cudnn.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Run Benchmarks

Set `platform_label` to `brev_l4`, `dgx_spark`, or a local test label before running on each machine.

In [ ]:
platform_label = 'local'
output_csv = f'../results/{platform_label}.csv'
!python3 benchmark_cudnn.py --platform {platform_label} --output {output_csv} --iters 30

## 4. Plot This Run

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(output_csv)
ax = df.pivot(index='mode', columns='benchmark', values='latency_ms').plot(kind='bar', figsize=(10, 5))
ax.set_ylabel('Median latency (ms, lower is better)')
ax.set_title(f'cuDNN benchmark on {platform_label}')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
df

## 5. Cross-Platform Comparison

After running on both Brev L4 and DGX Spark, this cell loads both CSV files and creates the headline comparison chart.

In [ ]:
result_files = [Path('../results/brev_l4.csv'), Path('../results/dgx_spark.csv')]
existing = [path for path in result_files if path.exists()]

if len(existing) < 2:
    print('Run the benchmark on both platforms first:', result_files)
else:
    combined = pd.concat([pd.read_csv(path) for path in existing], ignore_index=True)
    subset = combined[combined['benchmark'] == 'resnet18']
    ax = subset.pivot(index='mode', columns='platform_label', values='throughput_img_s').plot(kind='bar', figsize=(10, 5))
    ax.set_ylabel('Throughput (images/sec, higher is better)')
    ax.set_title('ResNet-18 inference throughput across platforms')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    display(subset)

## 6. Under the Hood

The optional C++ demo in `../cpp/fused_conv_bias_relu_demo.cu` shows the direct cuDNN idea behind framework-level fusion: a convolution, bias add, and ReLU can be represented as one fused cuDNN operation instead of separate framework calls.